In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
from xgboost import XGBRegressor
from sklearn.linear_model import LinearRegression

# Load and preprocess the data
data = pd.read_csv("data.csv")
data = data.drop(columns=['paper', 'catalyst used'])

# Convert numerical columns and clean
data['temperature(process)'] = pd.to_numeric(data['temperature(process)'], errors='coerce')
data['pressure(process)'] = pd.to_numeric(data['pressure(process)'], errors='coerce')

# Clean "conversion rate1" to extract numeric values
def clean_conversion_rate(value):
    if isinstance(value, str):
        value = value.replace('%', '').strip()
        if "to" in value:
            return np.mean([float(v) for v in value.split('to')])
        try:
            return float(value)
        except ValueError:
            return np.nan
    return value

data['conversion_rate'] = data['conversion rate1'].apply(clean_conversion_rate)
data = data.drop(columns=['conversion rate1'])

# Rename columns for consistency
data.columns = data.columns.str.strip().str.replace(' ', '_').str.replace('(process)', '').str.lower()

# Drop rows with missing values
data = data.dropna()

# Feature Engineering: Add interaction terms
data['temp_pressure_interaction'] = data['temperature'] * data['pressure']

# Split the dataset into features (X) and target (y)
X = data.drop(columns=['conversion_rate'])
y = data['conversion_rate']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardize the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Random Forest with Hyperparameter Tuning
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}

rf_grid = GridSearchCV(RandomForestRegressor(random_state=42), param_grid, scoring='neg_mean_squared_error', cv=5, verbose=1)
rf_grid.fit(X_train_scaled, y_train)
best_rf = rf_grid.best_estimator_

# XGBoost Model
xgb_model = XGBRegressor(random_state=42, n_estimators=200, max_depth=6, learning_rate=0.1)
xgb_model.fit(X_train_scaled, y_train)

# Predictions from both models
rf_pred_train = best_rf.predict(X_train_scaled)
xgb_pred_train = xgb_model.predict(X_train_scaled)

rf_pred_test = best_rf.predict(X_test_scaled)
xgb_pred_test = xgb_model.predict(X_test_scaled)

# Combine predictions (Meta-Model Approach)
# Use predictions from Random Forest and XGBoost as inputs to a Linear Regression meta-model
meta_X_train = np.column_stack((rf_pred_train, xgb_pred_train))
meta_X_test = np.column_stack((rf_pred_test, xgb_pred_test))

# Train the meta-model
meta_model = LinearRegression()
meta_model.fit(meta_X_train, y_train)

# Make final predictions using the meta-model
hybrid_pred = meta_model.predict(meta_X_test)

# Evaluate the hybrid model
hybrid_mse = mean_squared_error(y_test, hybrid_pred)
hybrid_r2 = r2_score(y_test, hybrid_pred)

print("\nHybrid Model (Random Forest + XGBoost):")
print("MSE:", hybrid_mse)
print("R2:", hybrid_r2)



Fitting 5 folds for each of 36 candidates, totalling 180 fits

Hybrid Model (Random Forest + XGBoost):
MSE: 2404.493201955868
R2: -0.48888254665325115

Predicted Conversion Rate (Hybrid Model): 54.210435927041296
Catalyst Used: ZrO2, WOx, and MoO3


In [2]:
# Predict conversion rate for new input
temperature = 250  # Example: Temperature in °C
pressure = 2   # Example: Pressure in MPa
surface_area = 2382  # Example: Surface area in m²/g
reaction_time = 4   # Example: Reaction time in hours
lignin_loading = 0.5  # Example: Lignin loading in wt%
catalyst = "ZrO2, WOx, and MoO3"

# Add interaction term for prediction
temp_pressure_interaction = temperature * pressure

# Prepare input as DataFrame
input_data = pd.DataFrame([[temperature, pressure, surface_area, reaction_time, lignin_loading, temp_pressure_interaction]],
                          columns=X_train.columns)

# Scale the input
input_data_scaled = scaler.transform(input_data)

# Predict with both models
rf_input_pred = best_rf.predict(input_data_scaled)
xgb_input_pred = xgb_model.predict(input_data_scaled)

# Combine predictions for the meta-model
input_meta_features = np.column_stack((rf_input_pred, xgb_input_pred))
final_prediction = meta_model.predict(input_meta_features)

print("\nPredicted Conversion Rate (Hybrid Model):", final_prediction[0])
print("Catalyst Used:", catalyst)



Predicted Conversion Rate (Hybrid Model): 19.20275128010161
Catalyst Used: ZrO2, WOx, and MoO3
